<a href="https://colab.research.google.com/github/AbdulBasit-5/hotel-model/blob/main/Hotel_Notebook1_Baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hotel Cancellation Research — Notebook 1: Data check, reproduction, and honest baselines

This notebook covers **Weeks 1–4** of the research plan (Stage 1: *will this booking be cancelled?*):

1. Load the Hotel Booking Demand dataset and **verify row counts**.
2. Inspect the known traps: **duplicate rows** and **columns that leak the answer**.
3. Run baselines under three protocols, for each hotel separately:
   - **P1 — random 80:20 on raw data** (closest guess to the target paper; check its full text to confirm)
   - **P2 — random 80:20 after removing duplicates**
   - **P3 — time-based split** (train on earlier arrivals, test on the latest 20%) after removing duplicates
4. Optional: reproduce **TabNet** (the target paper's model).
5. **Gate 1 check** against the paper: AUC-ROC 0.862 (resort, H1) and 0.846 (city, H2).

**How to run:** `Runtime → Run all`. The free CPU runtime is enough. The main experiment takes roughly 15–30 minutes; set `SEEDS = [0]` for a quick first run.

Target paper: Duong & Phan (2026), *Information Technology & Tourism* 28:54 — https://link.springer.com/article/10.1007/s40558-026-00394-y

## 1. Setup

In [1]:
!pip -q install catboost xgboost lightgbm pytorch-tabnet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 3.1 MB/s eta 0:00:00


In [ ]:
import os, time, warnings, urllib.request
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             accuracy_score, brier_score_loss)
warnings.filterwarnings('ignore')

TARGET = 'is_canceled'
ALWAYS_DROP = ['reservation_status', 'reservation_status_date', 'is_canceled', 'hotel',
               'arrival_date_year', 'arrival_date']
RISKY = ['assigned_room_type', 'booking_changes', 'days_in_waiting_list']
CAT_ALL = ['arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel',
           'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type', 'agent', 'company']

SEEDS = [0, 1, 2, 3, 4]        # use [0] for a quick first run
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)


## 2. Load the dataset and verify it

The file downloads from the TidyTuesday mirror on GitHub (same data as the Kaggle `hotel_bookings.csv`).
If the download fails, download `hotel_bookings.csv` from https://www.kaggle.com/jessemostipak/hotel-booking-demand,
upload it to Colab (folder icon on the left), and re-run this cell.

In [3]:
FILE = 'hotel_bookings.csv'
URL = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv'
if not os.path.exists(FILE):
    try:
        urllib.request.urlretrieve(URL, FILE)
        print('downloaded', FILE)
    except Exception as e:
        print('FAILED to download ->', e, '\nUpload hotel_bookings.csv manually and re-run.')

df = pd.read_csv(FILE)
print(f'rows={len(df):,d} (expected 119,390)   columns={df.shape[1]} (expected 32)')
print(df['hotel'].value_counts().rename('rows'), '\n(expected: City Hotel 79,330 · Resort Hotel 40,060)')
print(f"\nOverall cancellation rate: {df['is_canceled'].mean():.1%}")

downloaded hotel_bookings.csv
rows=119,390 (expected 119,390)   columns=32 (expected 32)
hotel
City Hotel      79330
Resort Hotel    40060
Name: rows, dtype: int64 
(expected: City Hotel 79,330 · Resort Hotel 40,060)

Overall cancellation rate: 37.0%


## Dataset Verification Report

In [9]:
# Consolidated Dataset Information
print('--- General Dataset Information ---')
print(f"Dataset: Hotel Booking Demand")
print(f"Total rows: {len(df):,d} (expected 119,390)")
print(f"Total columns: {df.shape[1]} (expected 32)")
print(f"Target variable: '{TARGET}'")

print('\n--- Hotel Distribution ---')
print(df['hotel'].value_counts().rename('rows'))
print('(Expected: City Hotel 79,330 · Resort Hotel 40,060)')

print('\n--- Overall Cancellation Rate ---')
print(f"Overall cancellation rate: {df['is_canceled'].mean():.1%}")

# Date Range Verification (already calculated in -ltF9FKLhOxm, display for report)
MONTHS = {m: i for i, m in enumerate(['January','February','March','April','May','June','July',
                                       'August','September','October','November','December'], 1)}
df['arrival_date'] = pd.to_datetime(dict(year=df['arrival_date_year'],
                                         month=df['arrival_date_month'].map(MONTHS),
                                         day=df['arrival_date_day_of_month']))
print('\n--- Dataset Period (Arrival Dates) ---')
print(f"Arrival dates: {df['arrival_date'].min().date()} to {df['arrival_date'].max().date()}")
print("(Expected: July 2015 to August 2017)")


--- General Dataset Information ---
Dataset: Hotel Booking Demand
Total rows: 119,390 (expected 119,390)
Total columns: 33 (expected 32)
Target variable: 'is_canceled'

--- Hotel Distribution ---
hotel
City Hotel      79330
Resort Hotel    40060
Name: rows, dtype: int64
(Expected: City Hotel 79,330 · Resort Hotel 40,060)

--- Overall Cancellation Rate ---
Overall cancellation rate: 37.0%

--- Dataset Period (Arrival Dates) ---
Arrival dates: 2015-07-01 to 2017-08-31
(Expected: July 2015 to August 2017)


In [10]:
# Missing Values Check
print('--- Missing Values Summary ---')
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

if not missing_values.empty:
    missing_percent = (missing_values / len(df)) * 100
    missing_df = pd.DataFrame({'Missing Count': missing_values, 'Missing %': missing_percent.round(2)})
    display(missing_df)
    missing_df.to_csv(f'{RESULTS_DIR}/missing_values_summary.csv')
    print('\nMissing value handling strategy for these columns: Imputation handled by `SimpleImputer` within preprocessing pipelines.')
else:
    print('No missing values found in the dataset.')


--- Missing Values Summary ---


,Missing Count,Missing %
company,112593,94.31
agent,16340,13.69
country,488,0.41
children,4,0.00



Missing value handling strategy for these columns: Imputation handled by `SimpleImputer` within preprocessing pipelines.


In [ ]:
# Duplicate Rows Check
print('\n--- Duplicate Rows Summary ---')
dup = df.duplicated()
print(f'Exact duplicate rows: {dup.sum():,d} of {len(df):,d} ({dup.mean():.1%})')

df_dedup = df[~dup].copy()
summary = pd.DataFrame({
    'rows_raw': df.groupby('hotel').size(),
    'cancel_rate_raw': df.groupby('hotel')['is_canceled'].mean(),
    'rows_dedup': df_dedup.groupby('hotel').size(),
    'cancel_rate_dedup': df_dedup.groupby('hotel')['is_canceled'].mean(),
}).round(3)

print('\n--- Deduplication Statistics ---')
display(summary)
print('\nDuplicate rows are handled by running separate experiments on raw data (P1) and deduplicated data (P2, P3), as per Part 4.')



--- Duplicate Rows Summary ---
Exact duplicate rows: 31,994 of 119,390 (26.8%)

--- Deduplication Statistics (from cell -ltF9FKLhOxm) ---


,rows_raw,cancel_rate_raw,rows_dedup,cancel_rate_dedup
hotel,,,,
City Hotel,79330,0.417,53428,0.300
Resort Hotel,40060,0.278,33968,0.235



Duplicate rows are handled by running separate experiments on raw data (P1) and deduplicated data (P2, P3), as per Part 4.


In [12]:
# Target Distribution
print('\n--- Target (`is_canceled`) Distribution ---')
target_dist = df[TARGET].value_counts(normalize=True).mul(100).rename('Percentage').to_frame()
display(target_dist)
target_dist.to_csv(f'{RESULTS_DIR}/target_distribution.csv')



--- Target (`is_canceled`) Distribution ---


,Percentage
is_canceled,
0,62.958372
1,37.041628


In [13]:
# Categorical and Numerical Columns

# Redefine CAT_ALL and use the feature_lists function to get active features.
# This ensures we are getting the features actually used in the models.

# First, ensure 'agent' and 'company' are preprocessed before determining types dynamically if needed
# or rely on the CAT_ALL definition and feature_lists helper.

# Assuming df is the original dataframe for inspection
# Drop columns that are always excluded or are the target for this analysis
initial_inspect_df = df.drop(columns=ALWAYS_DROP + [TARGET], errors='ignore')

# Dynamically infer types if not relying purely on CAT_ALL
# For this notebook, CAT_ALL and the feature_lists function are the primary source

active_feats, active_cat, active_num = feature_lists(drop_risky=False)

print('\n--- Categorical Features (used in models) ---')
print(active_cat)

print('\n--- Numerical Features (used in models) ---')
print(active_num)



--- Categorical Features (used in models) ---
['arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type', 'agent', 'company']

--- Numerical Features (used in models) ---
['lead_time', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']


### Impossible or Suspicious Values

Initial inspection of the dataset schema (`df.info()`, `df.describe()`) and value distributions did not immediately reveal impossible values that would require special cleaning beyond standard imputation. For instance, `adults`, `children`, `babies` can be 0, indicating no guests, which is plausible in hotel bookings (e.g., for booking management or placeholders). No explicit values like negative guest counts or dates far outside expected ranges were identified.

Further in-depth domain-specific checks could be added if more context on 'impossible' values is provided (e.g., maximum plausible number of guests, minimum stay duration, etc.).

## 3. Build arrival dates and check duplicates

The dataset has no booking IDs, so many rows are exact copies. If copies land in both the training
and test sets, the model is tested on bookings it has already seen, which inflates scores.

In [4]:
MONTHS = {m: i for i, m in enumerate(['January','February','March','April','May','June','July',
                                       'August','September','October','November','December'], 1)}
df['arrival_date'] = pd.to_datetime(dict(year=df['arrival_date_year'],
                                         month=df['arrival_date_month'].map(MONTHS),
                                         day=df['arrival_date_day_of_month']))
print('Arrival dates:', df['arrival_date'].min().date(), 'to', df['arrival_date'].max().date())

dup = df.drop(columns=['arrival_date']).duplicated()
print(f'\nExact duplicate rows: {dup.sum():,d} of {len(df):,d} ({dup.mean():.1%})')
df_dedup = df[~dup].copy()
summary = pd.DataFrame({
    'rows_raw': df.groupby('hotel').size(),
    'cancel_rate_raw': df.groupby('hotel')['is_canceled'].mean(),
    'rows_dedup': df_dedup.groupby('hotel').size(),
    'cancel_rate_dedup': df_dedup.groupby('hotel')['is_canceled'].mean(),
}).round(3)
summary

Arrival dates: 2015-07-01 to 2017-08-31

Exact duplicate rows: 31,994 of 119,390 (26.8%)


,rows_raw,cancel_rate_raw,rows_dedup,cancel_rate_dedup
hotel,,,,
City Hotel,79330,0.417,53428,0.300
Resort Hotel,40060,0.278,33968,0.235


## 4. Leakage check: which columns give away the answer?

`reservation_status` literally says *Canceled* / *Check-Out* / *No-Show*, and `reservation_status_date`
is the cancellation date for cancelled bookings. **Both are always removed from the inputs.**

The table below shows how perfectly `reservation_status` predicts the target.

In [5]:
pd.crosstab(df['reservation_status'], df['is_canceled'])

is_canceled,0,1
reservation_status,,
Canceled,0,43017
Check-Out,75166,0
No-Show,0,1207


## 5. Feature sets and preprocessing

- **Always removed:** `reservation_status`, `reservation_status_date` (leak the answer), `is_canceled` (the target),
  `hotel` (we model each hotel separately), `arrival_date_year` (a year value never seen in training breaks a time-based split),
  `arrival_date` (helper column).
- **Risky** (may only be known close to arrival): `assigned_room_type`, `booking_changes`, `days_in_waiting_list`.
  Main runs keep them (likely closer to the paper); an ablation in section 9 removes them.

Preprocessing is fitted on training data only, inside a Pipeline.

In [6]:
TARGET = 'is_canceled'
ALWAYS_DROP = ['reservation_status', 'reservation_status_date', 'is_canceled', 'hotel',
               'arrival_date_year', 'arrival_date']
RISKY = ['assigned_room_type', 'booking_changes', 'days_in_waiting_list']

CAT_ALL = ['arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel',
           'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type', 'agent', 'company']

def prepare(frame):
    """Return a copy with ID-like columns (agent, company) treated as text categories."""
    f = frame.copy()
    for c in ['agent', 'company']:
        f[c] = f[c].fillna(-1).astype(int).astype(str)
    return f

def feature_lists(drop_risky=False):
    feats = [c for c in df.columns if c not in ALWAYS_DROP and not (drop_risky and c in RISKY)]
    cat = [c for c in CAT_ALL if c in feats]
    num = [c for c in feats if c not in cat]
    return feats, cat, num

def make_preprocessor(cat, num, one_hot):
    if one_hot:   # for logistic regression
        cat_step = Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')),
                             ('enc', OneHotEncoder(handle_unknown='ignore', min_frequency=20))])
        num_step = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])
    else:         # for tree models
        cat_step = Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')),
                             ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))])
        num_step = SimpleImputer(strategy='median')
    return ColumnTransformer([('cat', cat_step, cat), ('num', num_step, num)])

def make_models(seed):
    from xgboost import XGBClassifier
    from lightgbm import LGBMClassifier
    from catboost import CatBoostClassifier
    return {
        'LogisticRegression': (LogisticRegression(max_iter=2000), True),
        'RandomForest': (RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=seed), False),
        'XGBoost': (XGBClassifier(n_estimators=400, max_depth=8, learning_rate=0.05, tree_method='hist',
                                  n_jobs=-1, random_state=seed, verbosity=0), False),
        'LightGBM': (LGBMClassifier(n_estimators=400, learning_rate=0.05, random_state=seed, verbose=-1), False),
        'CatBoost': (CatBoostClassifier(iterations=600, random_seed=seed, verbose=0, thread_count=-1), False),
    }

def evaluate(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {'auc_roc': roc_auc_score(y_true, proba),
            'auc_pr': average_precision_score(y_true, proba),
            'f1': f1_score(y_true, pred),
            'accuracy': accuracy_score(y_true, pred),
            'brier': brier_score_loss(y_true, proba)}

In [14]:
from sklearn.metrics import precision_score, recall_score

TARGET = 'is_canceled'
ALWAYS_DROP = ['reservation_status', 'reservation_status_date', 'is_canceled', 'hotel',
               'arrival_date_year', 'arrival_date']
RISKY = ['assigned_room_type', 'booking_changes', 'days_in_waiting_list']

CAT_ALL = ['arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel',
           'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type', 'agent', 'company']

def prepare(frame):
    """Return a copy with ID-like columns (agent, company) treated as text categories."""
    f = frame.copy()
    for c in ['agent', 'company']:
        f[c] = f[c].fillna(-1).astype(int).astype(str)
    return f

def feature_lists(drop_risky=False):
    feats = [c for c in df.columns if c not in ALWAYS_DROP and not (drop_risky and c in RISKY)]
    cat = [c for c in CAT_ALL if c in feats]
    num = [c for c in feats if c not in cat]
    return feats, cat, num

def make_preprocessor(cat, num, one_hot):
    if one_hot:   # for logistic regression
        cat_step = Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')),
                             ('enc', OneHotEncoder(handle_unknown='ignore', min_frequency=20))])
        num_step = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])
    else:         # for tree models
        cat_step = Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')),
                             ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))])
        num_step = SimpleImputer(strategy='median')
    return ColumnTransformer([('cat', cat_step, cat), ('num', num_step, num)])

def make_models(seed):
    from xgboost import XGBClassifier
    from lightgbm import LGBMClassifier
    from catboost import CatBoostClassifier
    return {
        'LogisticRegression': (LogisticRegression(max_iter=2000), True),
        'RandomForest': (RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=seed), False),
        'XGBoost': (XGBClassifier(n_estimators=400, max_depth=8, learning_rate=0.05, tree_method='hist',
                                  n_jobs=-1, random_state=seed, verbosity=0), False),
        'LightGBM': (LGBMClassifier(n_estimators=400, learning_rate=0.05, random_state=seed, verbose=-1), False),
        'CatBoost': (CatBoostClassifier(iterations=600, random_seed=seed, verbose=0, thread_count=-1), False),
    }

def evaluate(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {'auc_roc': roc_auc_score(y_true, proba),
            'auc_pr': average_precision_score(y_true, proba),
            'f1': f1_score(y_true, pred),
            'accuracy': accuracy_score(y_true, pred),
            'precision': precision_score(y_true, pred),
            'recall': recall_score(y_true, pred),
            'brier': brier_score_loss(y_true, proba)}


## 6. The three protocols

In [7]:
HOTELS = {'H1 (Resort)': 'Resort Hotel', 'H2 (City)': 'City Hotel'}

def split(protocol, hotel_name, seed):
    if protocol == 'P1 random, raw':
        data = prepare(df[df.hotel == hotel_name])
        return train_test_split(data, test_size=0.2, stratify=data[TARGET], random_state=seed)
    if protocol == 'P2 random, dedup':
        data = prepare(df_dedup[df_dedup.hotel == hotel_name])
        return train_test_split(data, test_size=0.2, stratify=data[TARGET], random_state=seed)
    if protocol == 'P3 time-based, dedup':
        data = prepare(df_dedup[df_dedup.hotel == hotel_name]).sort_values('arrival_date')
        cut = int(len(data) * 0.8)
        return data.iloc[:cut], data.iloc[cut:]
    raise ValueError(protocol)

PROTOCOLS = ['P1 random, raw', 'P2 random, dedup', 'P3 time-based, dedup']
for h_label, h_name in HOTELS.items():
    tr, te = split('P3 time-based, dedup', h_name, 0)
    print(f"{h_label}: time-based split trains on arrivals up to {tr.arrival_date.max().date()}, "
          f"tests on {te.arrival_date.min().date()} to {te.arrival_date.max().date()} ({len(te):,d} rows)")

H1 (Resort): time-based split trains on arrivals up to 2017-04-22, tests on 2017-04-22 to 2017-08-31 (6,794 rows)
H2 (City): time-based split trains on arrivals up to 2017-05-06, tests on 2017-05-06 to 2017-08-31 (10,686 rows)


## 7. Main experiment: 5 models × 3 protocols × 2 hotels × seeds

In [18]:
FEATS, CAT, NUM = feature_lists(drop_risky=False)
rows = []
for h_label, h_name in HOTELS.items():
    for protocol in PROTOCOLS:
        for seed in SEEDS:
            tr, te = split(protocol, h_name, seed)
            for name, (model, one_hot) in make_models(seed).items():
                pipe = Pipeline([('prep', make_preprocessor(CAT, NUM, one_hot)), ('model', model)])
                t0 = time.time(); pipe.fit(tr[FEATS], tr[TARGET]); fit_s = time.time() - t0
                proba = pipe.predict_proba(te[FEATS])[:, 1]
                rows.append({'hotel': h_label, 'protocol': protocol, 'model': name, 'seed': seed,
                             'fit_seconds': fit_s, **evaluate(te[TARGET].values, proba)})
            print(f'{h_label:12s} {protocol:22s} seed {seed} done')
res = pd.DataFrame(rows)

H1 (Resort)  P1 random, raw         seed 0 done
H1 (Resort)  P1 random, raw         seed 1 done
H1 (Resort)  P1 random, raw         seed 2 done
H1 (Resort)  P1 random, raw         seed 3 done
H1 (Resort)  P1 random, raw         seed 4 done
H1 (Resort)  P2 random, dedup       seed 0 done
H1 (Resort)  P2 random, dedup       seed 1 done
H1 (Resort)  P2 random, dedup       seed 2 done
H1 (Resort)  P2 random, dedup       seed 3 done
H1 (Resort)  P2 random, dedup       seed 4 done
H1 (Resort)  P3 time-based, dedup   seed 0 done
H1 (Resort)  P3 time-based, dedup   seed 1 done
H1 (Resort)  P3 time-based, dedup   seed 2 done
H1 (Resort)  P3 time-based, dedup   seed 3 done
H1 (Resort)  P3 time-based, dedup   seed 4 done
H2 (City)    P1 random, raw         seed 0 done
H2 (City)    P1 random, raw         seed 1 done
H2 (City)    P1 random, raw         seed 2 done
H2 (City)    P1 random, raw         seed 3 done
H2 (City)    P1 random, raw         seed 4 done
H2 (City)    P2 random, dedup       seed

### Results (mean ± std over seeds)

In [ ]:
summary_main = (res.groupby(['hotel', 'protocol', 'model'])[['auc_roc', 'auc_pr', 'f1', 'brier']]
                .agg(['mean', 'std']).round(4))
summary_main

In [19]:
summary_main = (res.groupby(['hotel', 'protocol', 'model'])[['auc_roc', 'auc_pr', 'f1', 'accuracy', 'precision', 'recall', 'brier']]
                .agg(['mean', 'std']).round(4))
summary_main

auc_roc          auc_pr  \
                                                       mean     std    mean   
hotel       protocol             model                                        
H1 (Resort) P1 random, raw       CatBoost            0.9591  0.0015  0.9090   
                                 LightGBM            0.9612  0.0014  0.9124   
                                 LogisticRegression  0.9227  0.0023  0.8326   
                                 RandomForest        0.9611  0.0010  0.9175   
                                 XGBoost             0.9626  0.0016  0.9163   
            P2 random, dedup     CatBoost            0.9425  0.0026  0.8364   
                                 LightGBM            0.9448  0.0028  0.8395   
                                 LogisticRegression  0.9018  0.0041  0.7369   
                                 RandomForest        0.9379  0.0028  0.8198   
                                 XGBoost             0.9450  0.0024  0.8411   
            P3 time-based, dedup CatBoost            0.8911  0.0018  0.7799   
                                 LightGBM            0.8866  0.0000  0.7768   
                                 LogisticRegression  0.8889  0.0000  0.7621   
                                 RandomForest        0.8736  0.0013  0.7571   
                                 XGBoost             0.8878  0.0000  0.7756   
H2 (City)   P1 random, raw       CatBoost            0.9486  0.0010  0.9384   
                                 LightGBM            0.9482  0.0010  0.9376   
                                 LogisticRegression  0.9078  0.0011  0.8879   
                                 RandomForest        0.9573  0.0015  0.9498   
                                 XGBoost             0.9515  0.0007  0.9417   
            P2 random, dedup     CatBoost            0.8969  0.0029  0.7977   
                                 LightGBM            0.8969  0.0033  0.7959   
                                 LogisticRegression  0.8394  0.0035  0.6817   
                                 RandomForest        0.8895  0.0034  0.7833   
                                 XGBoost             0.8996  0.0023  0.8018   
            P3 time-based, dedup CatBoost            0.8360  0.0010  0.7110   
                                 LightGBM            0.8336  0.0000  0.7079   
                                 LogisticRegression  0.8059  0.0000  0.6865   
                                 RandomForest        0.8129  0.0006  0.6786   
                                 XGBoost             0.8308  0.0000  0.7022   

                                                                 f1          \
                                                        std    mean     std   
hotel       protocol             model                                        
H1 (Resort) P1 random, raw       CatBoost            0.0042  0.8175  0.0065   
                                 LightGBM            0.0037  0.8206  0.0053   
                                 LogisticRegression  0.0063  0.7305  0.0020   
                                 RandomForest        0.0024  0.8263  0.0037   
                                 XGBoost             0.0041  0.8259  0.0047   
            P2 random, dedup     CatBoost            0.0039  0.7505  0.0082   
                                 LightGBM            0.0050  0.7530  0.0098   
                                 LogisticRegression  0.0086  0.6445  0.0066   
                                 RandomForest        0.0046  0.7429  0.0063   
                                 XGBoost             0.0037  0.7583  0.0078   
            P3 time-based, dedup CatBoost            0.0027  0.5642  0.0044   
                                 LightGBM            0.0000  0.5725  0.0000   
                                 LogisticRegression  0.0000  0.5936  0.0000   
                                 RandomForest        0.0020  0.5540  0.0026   
                                 XGBoost             0.0000  0.5755  0.0000   
H2 (City)   P1 random,

### Headline table: best model AUC-ROC under each protocol

In [ ]:
best = (res.groupby(['hotel', 'protocol', 'model'])['auc_roc'].mean().reset_index()
        .sort_values('auc_roc', ascending=False).groupby(['hotel', 'protocol']).head(1)
        .pivot(index='hotel', columns='protocol', values=['auc_roc', 'model']))
best

## 8. Optional: reproduce TabNet (the target paper's model)

TabNet is slower; it runs on P1 with one seed by default. A GPU runtime helps
(`Runtime → Change runtime type → T4 GPU`) but is not required.

In [20]:
RUN_TABNET = True
TABNET_SEEDS = [0]
tab_rows = []
if RUN_TABNET:
    from pytorch_tabnet.tab_model import TabNetClassifier   # installed in the setup cell
    for h_label, h_name in HOTELS.items():
        for seed in TABNET_SEEDS:
            tr, te = split('P1 random, raw', h_name, seed)
            tr_fit, tr_val = train_test_split(tr, test_size=0.1, stratify=tr[TARGET], random_state=seed)
            prep = make_preprocessor(CAT, NUM, one_hot=False).fit(tr_fit[FEATS])
            Xf, Xv, Xt = (prep.transform(d[FEATS]).astype(np.float32) for d in (tr_fit, tr_val, te))
            clf = TabNetClassifier(seed=seed, verbose=0)
            clf.fit(Xf, tr_fit[TARGET].values, eval_set=[(Xv, tr_val[TARGET].values)],
                    eval_metric=['auc'], max_epochs=100, patience=15, batch_size=1024)
            proba = clf.predict_proba(Xt)[:, 1]
            tab_rows.append({'hotel': h_label, 'protocol': 'P1 random, raw', 'model': 'TabNet',
                             'seed': seed, **evaluate(te[TARGET].values, proba)})
            print(h_label, 'TabNet AUC-ROC =', round(tab_rows[-1]['auc_roc'], 4))
tab_res = pd.DataFrame(tab_rows)

Stop training because you reached max_epochs = 100 with best_epoch = 96 and best_val_0_auc = 0.9456
H1 (Resort) TabNet AUC-ROC = 0.9428
Stop training because you reached max_epochs = 100 with best_epoch = 94 and best_val_0_auc = 0.93978
H2 (City) TabNet AUC-ROC = 0.9299


## 9. Ablation: remove the risky columns (time-based split, CatBoost)

In [21]:
FEATS_S, CAT_S, NUM_S = feature_lists(drop_risky=True)
abl = []
from catboost import CatBoostClassifier
for h_label, h_name in HOTELS.items():
    tr, te = split('P3 time-based, dedup', h_name, 0)
    for variant, (F, C, N) in {'all features': (FEATS, CAT, NUM),
                               'without risky columns': (FEATS_S, CAT_S, NUM_S)}.items():
        pipe = Pipeline([('prep', make_preprocessor(C, N, False)),
                         ('model', CatBoostClassifier(iterations=600, random_seed=0, verbose=0))])
        pipe.fit(tr[F], tr[TARGET])
        abl.append({'hotel': h_label, 'features': variant,
                    **evaluate(te[TARGET].values, pipe.predict_proba(te[F])[:, 1])})
pd.DataFrame(abl).round(4)

,hotel,features,auc_roc,auc_pr,f1,accuracy,precision,recall,brier
0,H1 (Resort),all features,0.8880,0.7759,0.5623,0.7749,0.8043,0.4322,0.1567
1,H1 (Resort),without risky columns,0.8818,0.7551,0.5460,0.7663,0.7790,0.4203,0.1620
2,H2 (City),all features,0.8372,0.7133,0.6181,0.7602,0.6956,0.5561,0.1605
3,H2 (City),without risky columns,0.8246,0.6936,0.5912,0.7484,0.6823,0.5215,0.1681


## 10. Gate 1 check

Published (Duong & Phan 2026, TabNet): **AUC-ROC 0.862 on H1, 0.846 on H2** (AUC-PR 0.749 and 0.827).

- If your reproduced TabNet (section 8) is close to these, your setup matches theirs well enough.
- If your tuned-free tree models under P1 **already** reach or beat these numbers, that is an early sign
  your main improvement claim is realistic — but confirm the paper's exact split and columns first.

## 10. Gate 1 check and TabNet Comparison (Published vs. Our Results)

In [22]:
PAPER_AUC_ROC = {'H1 (Resort)': 0.862, 'H2 (City)': 0.846}
PAPER_AUC_PR = {'H1 (Resort)': 0.749, 'H2 (City)': 0.827}

comparison_rows = []

p1_results = res[res.protocol == 'P1 random, raw'].groupby(['hotel', 'model'])[['auc_roc', 'auc_pr']].mean().reset_index()

for h_label in HOTELS:
    # Get our best P1 model results
    best_p1_auc_roc_row = p1_results[p1_results.hotel == h_label].sort_values('auc_roc', ascending=False).iloc[0]
    best_p1_model_name = best_p1_auc_roc_row['model']
    our_best_p1_auc_roc = best_p1_auc_roc_row['auc_roc']

    # Get our TabNet results if available
    our_tabnet_auc_roc = np.nan
    our_tabnet_auc_pr = np.nan

    if len(tab_res) > 0: # Changed from if len(tab_res): to avoid checking length of empty dataframe if TabNet was skipped
        tabnet_h_results = tab_res[tab_res.hotel == h_label]
        if not tabnet_h_results.empty:
            our_tabnet_auc_roc = tabnet_h_results['auc_roc'].mean()
            our_tabnet_auc_pr = tabnet_h_results['auc_pr'].mean()

    # Compare AUC-ROC
    diff_auc_roc = our_tabnet_auc_roc - PAPER_AUC_ROC[h_label] if not np.isnan(our_tabnet_auc_roc) else np.nan
    diff_auc_pr = our_tabnet_auc_pr - PAPER_AUC_PR[h_label] if not np.isnan(our_tabnet_auc_pr) else np.nan

    comparison_rows.append({
        'Hotel': h_label,
        'Published AUC-ROC': PAPER_AUC_ROC[h_label],
        'Our TabNet AUC-ROC': our_tabnet_auc_roc,
        'AUC-ROC Difference': diff_auc_roc,
        'Published AUC-PR': PAPER_AUC_PR[h_label],
        'Our TabNet AUC-PR': our_tabnet_auc_pr,
        'AUC-PR Difference': diff_auc_pr,
        'Our Best P1 Model': best_p1_model_name,
        'Our Best P1 AUC-ROC': our_best_p1_auc_roc
    })

comparison_df = pd.DataFrame(comparison_rows).round(4)
display(comparison_df)

print('\nReminder: compare like with like. Confirm the paper used a random split and the same columns.')
print('Needs verification from the full paper for feature set, preprocessing, and exact split details.')

,Hotel,Published AUC-ROC,Our TabNet AUC-ROC,AUC-ROC Difference,Published AUC-PR,Our TabNet AUC-PR,AUC-PR Difference,Our Best P1 Model,Our Best P1 AUC-ROC
0,H1 (Resort),0.862,0.9428,0.0808,0.749,0.8776,0.1286,XGBoost,0.9626
1,H2 (City),0.846,0.9299,0.0839,0.827,0.9166,0.0896,RandomForest,0.9573



Reminder: compare like with like. Confirm the paper used a random split and the same columns.
Needs verification from the full paper for feature set, preprocessing, and exact split details.


The comparison above shows how our TabNet implementation and best P1 models perform against the published results.

**Note on comparison:** While our results provide a good baseline, direct comparison without verifying the exact feature set, preprocessing, and data splitting methodology used in the original paper requires caution. The stated differences should be interpreted with the understanding that subtle variations in these factors can lead to different outcomes. Further verification from the full paper is needed to ensure an 'apples-to-apples' comparison.

In [ ]:
PAPER = {'H1 (Resort)': 0.862, 'H2 (City)': 0.846}
p1 = res[res.protocol == 'P1 random, raw'].groupby(['hotel', 'model'])['auc_roc'].mean()
for h in HOTELS:
    best_model = p1[h].idxmax(); best_auc = p1[h].max()
    line = f'{h}: paper TabNet {PAPER[h]:.3f} | your best P1 model {best_model} {best_auc:.3f}'
    if len(tab_res):
        line += f" | your TabNet {tab_res[tab_res.hotel == h]['auc_roc'].mean():.3f}"
    print(line)
print('\nReminder: compare like with like. Confirm the paper used a random split and the same columns.')

## 11. Save everything

Download the CSV files from the `results/` folder (left sidebar).

## 11. Save everything

In [23]:
res.to_csv(f'{RESULTS_DIR}/main_experiment.csv', index=False)
summary_main.to_csv(f'{RESULTS_DIR}/summary_main.csv')
summary.to_csv(f'{RESULTS_DIR}/duplicates_summary.csv')
pd.DataFrame(abl).to_csv(f'{RESULTS_DIR}/ablation_risky_columns.csv', index=False)
if len(tab_res) > 0:
    tab_res.to_csv(f'{RESULTS_DIR}/tabnet_raw_results.csv', index=False)

# Save the comparison table
comparison_df.to_csv(f'{RESULTS_DIR}/tabnet_comparison.csv', index=False)

print('Saved:', os.listdir(RESULTS_DIR))

Saved: ['tabnet_comparison.csv', 'ablation_risky_columns.csv', 'summary_main.csv', 'main_experiment.csv', 'tabnet_raw_results.csv', 'duplicates_summary.csv', 'target_distribution.csv', 'missing_values_summary.csv']


In [24]:
res.to_csv(f'{RESULTS_DIR}/main_experiment.csv', index=False)
summary_main.to_csv(f'{RESULTS_DIR}/summary_main.csv')
summary.to_csv(f'{RESULTS_DIR}/duplicates_summary.csv')
pd.DataFrame(abl).to_csv(f'{RESULTS_DIR}/ablation_risky_columns.csv', index=False)
if len(tab_res):
    tab_res.to_csv(f'{RESULTS_DIR}/tabnet.csv', index=False)
print('Saved:', os.listdir(RESULTS_DIR))

Saved: ['tabnet_comparison.csv', 'tabnet.csv', 'ablation_risky_columns.csv', 'summary_main.csv', 'main_experiment.csv', 'tabnet_raw_results.csv', 'duplicates_summary.csv', 'target_distribution.csv', 'missing_values_summary.csv']
